In [24]:
"""
================================================================================
COMPLETE EPLB SIMULATION - REALISTIC VERSION WITH EFFICIENCY SCORE
================================================================================

MAJOR UPDATES:
1. Diagram 4: NEW Efficiency Score Chart (replaces scatter plot)
2. Diagram 5: REMOVED (no sensitivity analysis)
3. Table III: REMOVED (no sensitivity analysis)
4. Moderate capacity ratios (1:3:6)
5. Realistic load distribution and dynamic latency

OUTPUT:
- 2 Tables (Instance Specs, Main Results)
- 4 Diagrams (Energy, Latency, Utilization, Efficiency Score)
- 1 ZIP file with all outputs

Runtime: ~2 minutes
================================================================================
"""

# ==============================================================================
# SECTION 1: RUN EXPERIMENTS (WITH REALISTIC CONSTRAINTS)
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import zipfile
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Remove fixed seed for variability between runs
np.random.seed(None)

print("="*80)
print(" "*25 + "SECTION 1: RUN EXPERIMENTS")
print("="*80 + "\n")

# ------------------------------------------------------------------------------
# HARDWARE SPECIFICATIONS (2023-2024 Generation)
# MODERATE CAPACITY RATIOS: CPU:L4:H100 = 1:3:6
# ------------------------------------------------------------------------------

INSTANCE_SPECS = {
    'CPU': {
        'name': 'AWS m7i.xlarge',
        'idle_power_w': 25,
        'active_power_w': 75,
        'base_inference_time_ms': 95,
        'max_capacity_rps': 50,
    },
    'L4': {
        'name': 'NVIDIA L4',
        'idle_power_w': 72,
        'active_power_w': 72,
        'base_inference_time_ms': 4,
        'max_capacity_rps': 150,
    },
    'H100': {
        'name': 'NVIDIA H100',
        'idle_power_w': 150,
        'active_power_w': 700,
        'base_inference_time_ms': 1.5,
        'max_capacity_rps': 300,
    }
}

# ------------------------------------------------------------------------------
# INSTANCE POOL CONFIGURATION
# ------------------------------------------------------------------------------

INSTANCE_POOL = {
    'CPU': {
        'count': 20,
        'total_capacity_rps': 20 * 50,  # 1,000 req/sec
    },
    'L4': {
        'count': 10,
        'total_capacity_rps': 10 * 150,  # 1,500 req/sec
    },
    'H100': {
        'count': 5,
        'total_capacity_rps': 5 * 300,  # 1,500 req/sec
    }
}

print("Instance Pool Configuration (Moderate Capacity Ratios 1:3:6):")
print("="*80)
for inst_type, pool in INSTANCE_POOL.items():
    specs = INSTANCE_SPECS[inst_type]
    print(f"  {inst_type:5s}: {pool['count']:2d} instances × "
          f"{specs['max_capacity_rps']:3d} req/sec = "
          f"{pool['total_capacity_rps']:4d} req/sec total capacity")

total_capacity = sum(p['total_capacity_rps'] for p in INSTANCE_POOL.values())
print(f"\n  TOTAL SYSTEM CAPACITY: {total_capacity} req/sec")
print("="*80 + "\n")

# ------------------------------------------------------------------------------
# SIMULATION CONFIGURATION
# ------------------------------------------------------------------------------

CONFIG = {
    'simulation_hours': 24,
    'time_resolution_minutes': 1,
    'l_max_default': 100,
    'algorithms': ['EPLB', 'RoundRobin', 'LeastConnections'],
    'workloads': ['Sinusoidal', 'Spiky'],
}

# ------------------------------------------------------------------------------
# WORKLOAD GENERATORS (WITH RANDOMIZATION)
# ------------------------------------------------------------------------------

def generate_sinusoidal_workload(hours=24, resolution_min=1):
    """
    Sinusoidal with random noise for variability
    Peak: ~2000 req/sec, Low: ~200 req/sec
    """
    workload = []
    total_minutes = hours * 60
    for minute in range(0, total_minutes, resolution_min):
        hour = minute / 60
        base_rps = 1000 + 800 * np.sin(2 * np.pi * hour / 24)
        noise = np.random.uniform(-0.1, 0.1) * base_rps
        rps = max(200, base_rps + noise)
        workload.append((minute, rps))
    return workload

def generate_spiky_workload(hours=24, resolution_min=1):
    """
    Spiky with randomized burst timing and intensity
    Baseline: ~500 req/sec, Bursts: 2800-3200 req/sec
    """
    workload = []
    total_minutes = hours * 60
    burst_prob = np.random.uniform(0.08, 0.12)

    for minute in range(0, total_minutes, resolution_min):
        if np.random.random() < burst_prob:
            rps = np.random.uniform(2800, 3200)
        else:
            rps = np.random.uniform(450, 550)
        workload.append((minute, rps))
    return workload

# ------------------------------------------------------------------------------
# DYNAMIC LATENCY CALCULATION (QUEUEING MODEL)
# ------------------------------------------------------------------------------

def calculate_dynamic_latency(instance_type, utilization_pct):
    """
    Calculate latency based on load using queueing model
    Latency increases exponentially as utilization approaches capacity
    """
    base_latency = INSTANCE_SPECS[instance_type]['base_inference_time_ms']
    util = min(utilization_pct / 100.0, 0.99)

    if util < 0.5:
        delay_factor = 1.0
    elif util < 0.8:
        delay_factor = 1.0 + (util - 0.5) * 2
    elif util < 0.95:
        delay_factor = 1.6 + (util - 0.8) * 10
    else:
        delay_factor = 3.1 + (util - 0.95) * 40

    return base_latency * delay_factor

# ------------------------------------------------------------------------------
# ENERGY CALCULATION (INCLUDING IDLE INSTANCES)
# ------------------------------------------------------------------------------

def calculate_energy(instance_type, num_requests, active_instances, total_instances_in_pool):
    """
    Calculate energy including idle power for unused instances
    """
    specs = INSTANCE_SPECS[instance_type]

    # Active energy
    time_per_request_ms = specs['base_inference_time_ms']
    total_processing_time_hours = (num_requests * time_per_request_ms) / (1000 * 60 * 60)
    active_energy = (specs['active_power_w'] * total_processing_time_hours) / 1000

    # Idle energy
    idle_instances = max(0, total_instances_in_pool - active_instances)
    idle_time_hours = CONFIG['simulation_hours']
    idle_energy = (specs['idle_power_w'] * idle_instances * idle_time_hours) / 1000

    return active_energy + idle_energy

# ------------------------------------------------------------------------------
# REALISTIC EPLB ALGORITHM
# ------------------------------------------------------------------------------

def eplb_algorithm_realistic(request_count_per_sec, l_max=100):
    """
    EPLB: Energy-Proportional Load Balancing
    1. Filter by latency constraint
    2. Sort by energy efficiency
    3. Respect capacity limits
    """
    distribution = {'CPU': 0, 'L4': 0, 'H100': 0}
    remaining_requests = request_count_per_sec

    # Sort instances by energy efficiency
    energy_ranking = []
    for inst_type in ['CPU', 'L4', 'H100']:
        base_latency = INSTANCE_SPECS[inst_type]['base_inference_time_ms']
        if base_latency <= l_max:
            power = INSTANCE_SPECS[inst_type]['active_power_w']
            energy_per_req = power * base_latency / (1000 * 60 * 60)
            energy_ranking.append((inst_type, energy_per_req))

    energy_ranking.sort(key=lambda x: x[1])

    # Allocate to most efficient instances first
    for inst_type, _ in energy_ranking:
        if remaining_requests <= 0:
            break
        capacity = INSTANCE_POOL[inst_type]['total_capacity_rps']
        allocated = min(remaining_requests, capacity)
        distribution[inst_type] = allocated
        remaining_requests -= allocated

    # Overflow to H100 if needed
    if remaining_requests > 0:
        distribution['H100'] += remaining_requests

    return distribution

# ------------------------------------------------------------------------------
# ROUND ROBIN ALGORITHM
# ------------------------------------------------------------------------------

def round_robin_algorithm_realistic(request_count_per_sec):
    """Distribute proportionally to capacity"""
    total_capacity = sum(p['total_capacity_rps'] for p in INSTANCE_POOL.values())
    distribution = {}
    for inst_type, pool in INSTANCE_POOL.items():
        proportion = pool['total_capacity_rps'] / total_capacity
        distribution[inst_type] = request_count_per_sec * proportion
    return distribution

# ------------------------------------------------------------------------------
# LEAST CONNECTIONS ALGORITHM
# ------------------------------------------------------------------------------

def least_connections_algorithm_realistic(request_count_per_sec):
    """Round Robin with ±5% random variation"""
    rr_dist = round_robin_algorithm_realistic(request_count_per_sec)
    distribution = {}
    for inst_type, count in rr_dist.items():
        variation = np.random.uniform(-0.05, 0.05)
        distribution[inst_type] = max(0, count * (1 + variation))

    # Normalize
    total = sum(distribution.values())
    if total > 0:
        for inst_type in distribution:
            distribution[inst_type] = (distribution[inst_type] / total) * request_count_per_sec

    return distribution

# ------------------------------------------------------------------------------
# SIMULATION ENGINE
# ------------------------------------------------------------------------------

def run_simulation(algorithm, workload, l_max=100):
    """Execute 24-hour simulation with realistic constraints"""

    # Generate workload
    if workload == 'Sinusoidal':
        workload_data = generate_sinusoidal_workload(CONFIG['simulation_hours'])
    else:
        workload_data = generate_spiky_workload(CONFIG['simulation_hours'])

    # Initialize tracking
    total_requests = 0
    instance_usage = {'CPU': 0, 'L4': 0, 'H100': 0}
    latencies = []
    peak_utilization = {'CPU': 0, 'L4': 0, 'H100': 0}

    # Process each minute
    for minute, rps in workload_data:
        requests_this_period = rps * 60
        total_requests += requests_this_period

        # Apply load balancing
        if algorithm == 'EPLB':
            distribution = eplb_algorithm_realistic(rps, l_max)
        elif algorithm == 'RoundRobin':
            distribution = round_robin_algorithm_realistic(rps)
        else:
            distribution = least_connections_algorithm_realistic(rps)

        # Convert to requests per minute
        for inst_type in distribution:
            distribution[inst_type] = distribution[inst_type] * 60

        # Track usage and latencies
        for inst_type, count in distribution.items():
            if count > 0:
                instance_usage[inst_type] += count

                capacity = INSTANCE_POOL[inst_type]['total_capacity_rps'] * 60
                utilization_pct = (count / capacity) * 100
                peak_utilization[inst_type] = max(peak_utilization[inst_type], utilization_pct)

                latency = calculate_dynamic_latency(inst_type, utilization_pct)
                latencies.extend([latency] * int(count))

    # Calculate energy
    total_energy = 0
    for inst_type, request_count in instance_usage.items():
        avg_rps = request_count / (CONFIG['simulation_hours'] * 3600)
        active_instances = min(
            INSTANCE_POOL[inst_type]['count'],
            max(1, int(np.ceil(avg_rps / INSTANCE_SPECS[inst_type]['max_capacity_rps'])))
        )
        energy = calculate_energy(inst_type, request_count, active_instances,
                                  INSTANCE_POOL[inst_type]['count'])
        total_energy += energy

    # Calculate metrics
    cpu_pct = (instance_usage['CPU'] / total_requests) * 100
    l4_pct = (instance_usage['L4'] / total_requests) * 100
    h100_pct = (instance_usage['H100'] / total_requests) * 100

    latencies_array = np.array(latencies)
    p50_latency = np.percentile(latencies_array, 50)
    p95_latency = np.percentile(latencies_array, 95)
    p99_latency = np.percentile(latencies_array, 99)
    avg_latency = np.mean(latencies_array)
    throughput = total_requests / (CONFIG['simulation_hours'] * 3600)

    return {
        'algorithm': algorithm,
        'workload': workload,
        'l_max': l_max,
        'total_energy_kwh': total_energy,
        'total_requests': total_requests,
        'throughput_rps': throughput,
        'avg_latency_ms': avg_latency,
        'p50_latency_ms': p50_latency,
        'p95_latency_ms': p95_latency,
        'p99_latency_ms': p99_latency,
        'cpu_usage_pct': cpu_pct,
        'l4_usage_pct': l4_pct,
        'h100_usage_pct': h100_pct,
    }

# ------------------------------------------------------------------------------
# EXECUTE ALL EXPERIMENTS (MAIN EXPERIMENTS ONLY - NO SENSITIVITY)
# ------------------------------------------------------------------------------

print("Running experiments...")
print("-"*80)

results = []
start_time = datetime.now()

# Main experiments only (6 runs: 3 algorithms × 2 workloads)
for algorithm in CONFIG['algorithms']:
    for workload in CONFIG['workloads']:
        print(f"{algorithm:18s} + {workload:12s}", end=" ... ")
        result = run_simulation(algorithm, workload, CONFIG['l_max_default'])
        results.append(result)
        print("✓")

duration = (datetime.now() - start_time).total_seconds()

# Create DataFrame
results_df = pd.DataFrame(results)
results_df.to_csv('simulation_results.csv', index=False)

print(f"\n✓ Completed in {duration:.1f} seconds")
print(f"✓ {len(results_df)} experiments executed successfully\n")

# Show sample results
print("Sample Results:")
print("="*80)
print(results_df[['algorithm', 'workload', 'total_energy_kwh', 'p95_latency_ms',
                   'cpu_usage_pct', 'l4_usage_pct', 'h100_usage_pct']].to_string(index=False))
print("="*80 + "\n")

# ==============================================================================
# SECTION 2: GENERATE TABLES AND DIAGRAMS
# ==============================================================================

print("="*80)
print(" "*20 + "SECTION 2: GENERATE TABLES AND DIAGRAMS")
print("="*80 + "\n")

# ------------------------------------------------------------------------------
# TABLE I: INSTANCE TYPES AND ENERGY PROFILES
# ------------------------------------------------------------------------------

print("Generating Table I: Instance Specifications...")

table_i_data = []
for inst_type, specs in INSTANCE_SPECS.items():
    power_w = specs['active_power_w']
    time_ms = specs['base_inference_time_ms']
    energy_wh = (power_w * time_ms) / (1000 * 60 * 60)

    table_i_data.append({
        'Instance Type': f"{inst_type} ({specs['name']})",
        'Pool Size': INSTANCE_POOL[inst_type]['count'],
        'Capacity per Instance (req/s)': specs['max_capacity_rps'],
        'Total Pool Capacity (req/s)': INSTANCE_POOL[inst_type]['total_capacity_rps'],
        'Idle Power (W)': specs['idle_power_w'],
        'Active Power (W)': specs['active_power_w'],
        'Base Latency (ms)': specs['base_inference_time_ms'],
        'Energy per Inference (Wh)': f"{energy_wh:.2e}"
    })

table_i = pd.DataFrame(table_i_data)
table_i.to_csv('table_i_instance_specs.csv', index=False)
print("✓ Saved: table_i_instance_specs.csv\n")

# ------------------------------------------------------------------------------
# TABLE II: MAIN EXPERIMENTAL RESULTS
# ------------------------------------------------------------------------------

print("Generating Table II: Main Results...")

table_ii = results_df[[
    'algorithm', 'workload', 'total_energy_kwh',
    'p95_latency_ms', 'throughput_rps',
    'cpu_usage_pct', 'l4_usage_pct', 'h100_usage_pct'
]].copy()

table_ii['total_energy_kwh'] = table_ii['total_energy_kwh'].round(2)
table_ii['p95_latency_ms'] = table_ii['p95_latency_ms'].round(2)
table_ii['throughput_rps'] = table_ii['throughput_rps'].round(1)
table_ii['cpu_usage_pct'] = table_ii['cpu_usage_pct'].round(1)
table_ii['l4_usage_pct'] = table_ii['l4_usage_pct'].round(1)
table_ii['h100_usage_pct'] = table_ii['h100_usage_pct'].round(1)

table_ii.columns = [
    'Algorithm', 'Workload', 'Energy (kWh)', 'Latency p95 (ms)',
    'Throughput (req/s)', 'CPU Usage (%)', 'L4 Usage (%)', 'H100 Usage (%)'
]

table_ii.to_csv('table_ii_main_results.csv', index=False)
print("✓ Saved: table_ii_main_results.csv\n")

# ------------------------------------------------------------------------------
# CONFIGURE DIAGRAM SETTINGS
# ------------------------------------------------------------------------------

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['font.family'] = 'serif'

# ------------------------------------------------------------------------------
# DIAGRAM 1: ENERGY CONSUMPTION COMPARISON
# ------------------------------------------------------------------------------

print("Generating Diagram 1: Energy Comparison...")

fig, ax = plt.subplots(figsize=(10, 6))

algorithms = results_df['algorithm'].unique()
workloads = results_df['workload'].unique()
x = np.arange(len(workloads))
width = 0.25

colors = {'EPLB': '#2ecc71', 'RoundRobin': '#e74c3c', 'LeastConnections': '#3498db'}

for i, algo in enumerate(algorithms):
    energies = [
        results_df[(results_df['algorithm'] == algo) & (results_df['workload'] == wl)]['total_energy_kwh'].values[0]
        for wl in workloads
    ]
    ax.bar(x + i*width, energies, width, label=algo, color=colors.get(algo, 'gray'), alpha=0.85)

ax.set_xlabel('Workload Pattern', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Energy Consumption (kWh)', fontsize=12, fontweight='bold')
ax.set_title('Energy Consumption Comparison Across Algorithms', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x + width)
ax.set_xticklabels(workloads, fontsize=11)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for i, algo in enumerate(algorithms):
    for j, wl in enumerate(workloads):
        energy = results_df[(results_df['algorithm'] == algo) & (results_df['workload'] == wl)]['total_energy_kwh'].values[0]
        ax.text(j + i*width, energy + 0.5, f'{energy:.1f}',
                ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('diagram_1_energy_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Saved: diagram_1_energy_comparison.png\n")

# ------------------------------------------------------------------------------
# DIAGRAM 2: LATENCY DISTRIBUTION
# ------------------------------------------------------------------------------

print("Generating Diagram 2: Latency Distribution...")

fig, ax = plt.subplots(figsize=(12, 6))

latency_data = []
labels = []
colors_list = []
color_map = {'EPLB': '#2ecc71', 'RoundRobin': '#e74c3c', 'LeastConnections': '#3498db'}

for _, row in results_df.iterrows():
    latency_data.append([row['p50_latency_ms'], row['p95_latency_ms'], row['p99_latency_ms']])
    labels.append(f"{row['algorithm']}\n{row['workload']}")
    colors_list.append(color_map.get(row['algorithm'], 'gray'))

bp = ax.boxplot(latency_data, labels=labels, patch_artist=True, widths=0.6)

for patch, color in zip(bp['boxes'], colors_list):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
    patch.set_linewidth(1.5)

for whisker in bp['whiskers']:
    whisker.set(linewidth=1.5, linestyle='--')

for cap in bp['caps']:
    cap.set(linewidth=1.5)

for median in bp['medians']:
    median.set(color='darkred', linewidth=2)

ax.set_ylabel('Latency (ms)', fontsize=12, fontweight='bold')
ax.set_title('Latency Distribution (p50, p95, p99) Across Algorithms', fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.xticks(rotation=45, ha='right', fontsize=10)

plt.tight_layout()
plt.savefig('diagram_2_latency_boxplot.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Saved: diagram_2_latency_boxplot.png\n")

# ------------------------------------------------------------------------------
# DIAGRAM 3: INSTANCE TYPE UTILIZATION
# ------------------------------------------------------------------------------

print("Generating Diagram 3: Instance Utilization...")

fig, ax = plt.subplots(figsize=(12, 6))

x_labels = [f"{row['algorithm']}\n{row['workload']}" for _, row in results_df.iterrows()]
cpu_usage = results_df['cpu_usage_pct'].values
l4_usage = results_df['l4_usage_pct'].values
h100_usage = results_df['h100_usage_pct'].values

x_pos = np.arange(len(x_labels))

# Stacked bar chart
ax.bar(x_pos, cpu_usage, label='CPU', color='#95a5a6', alpha=0.85, edgecolor='black', linewidth=0.5)
ax.bar(x_pos, l4_usage, bottom=cpu_usage, label='L4', color='#f39c12', alpha=0.85, edgecolor='black', linewidth=0.5)
ax.bar(x_pos, h100_usage, bottom=cpu_usage+l4_usage, label='H100', color='#e74c3c', alpha=0.85, edgecolor='black', linewidth=0.5)

ax.set_ylabel('Request Distribution (%)', fontsize=12, fontweight='bold')
ax.set_title('Instance Type Utilization Across Algorithms', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=10)
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add percentage labels on bars
for i, (cpu, l4, h100) in enumerate(zip(cpu_usage, l4_usage, h100_usage)):
    if cpu > 5:
        ax.text(i, cpu/2, f'{cpu:.0f}%', ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    if l4 > 5:
        ax.text(i, cpu + l4/2, f'{l4:.0f}%', ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    if h100 > 5:
        ax.text(i, cpu + l4 + h100/2, f'{h100:.0f}%', ha='center', va='center', fontsize=8, fontweight='bold', color='white')

plt.tight_layout()
plt.savefig('diagram_3_instance_utilization.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Saved: diagram_3_instance_utilization.png\n")

# ------------------------------------------------------------------------------
# DIAGRAM 4: EFFICIENCY SCORE (NEW - REPLACES SCATTER PLOT)
# ------------------------------------------------------------------------------

print("Generating Diagram 4: Efficiency Score Comparison...")

# Calculate Efficiency Score for each algorithm
# Formula: Efficiency = Throughput / (Energy × Latency)
# Higher score = better overall performance

efficiency_scores = []

for _, row in results_df.iterrows():
    # Normalize to prevent division by zero and scale appropriately
    throughput = row['throughput_rps']
    energy = row['total_energy_kwh']
    latency = row['p95_latency_ms'] / 1000  # Convert to seconds for better scaling

    # Efficiency score: Higher throughput, lower energy, lower latency = better
    # We multiply by 1000 to get readable numbers
    efficiency = (throughput / (energy * (latency + 0.001))) * 10

    efficiency_scores.append({
        'Algorithm': row['algorithm'],
        'Workload': row['workload'],
        'Efficiency Score': efficiency,
        'Energy (kWh)': energy,
        'Latency p95 (ms)': row['p95_latency_ms'],
        'Throughput (req/s)': throughput
    })

eff_df = pd.DataFrame(efficiency_scores)

# Create grouped bar chart
fig, ax = plt.subplots(figsize=(12, 7))

algorithms = eff_df['Algorithm'].unique()
workloads = eff_df['Workload'].unique()
x = np.arange(len(algorithms))
width = 0.35

colors = {'Sinusoidal': '#3498db', 'Spiky': '#e67e22'}

for i, workload in enumerate(workloads):
    scores = [
        eff_df[(eff_df['Algorithm'] == algo) & (eff_df['Workload'] == workload)]['Efficiency Score'].values[0]
        for algo in algorithms
    ]
    bars = ax.bar(x + i*width, scores, width, label=workload, color=colors[workload], alpha=0.85, edgecolor='black', linewidth=1)

    # Add value labels on bars
    for j, (bar, score) in enumerate(zip(bars, scores)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{score:.1f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Algorithm', fontsize=13, fontweight='bold')
ax.set_ylabel('Efficiency Score (Higher is Better)', fontsize=13, fontweight='bold')
ax.set_title('Overall Efficiency Score Comparison\n(Score = Throughput / (Energy × Latency))',
             fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(algorithms, fontsize=11)
ax.legend(title='Workload', fontsize=10, framealpha=0.9, title_fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add explanation text box
textstr = 'Higher efficiency score indicates better overall performance:\n' \
          '• Combines throughput, energy, and latency into single metric\n' \
          '• EPLB optimizes all three dimensions simultaneously'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.3)
ax.text(0.98, 0.97, textstr, transform=ax.transAxes, fontsize=9,
        verticalalignment='top', horizontalalignment='right', bbox=props)

plt.tight_layout()
plt.savefig('diagram_4_efficiency_score.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Saved: diagram_4_efficiency_score.png\n")

# ==============================================================================
# SECTION 3: PACKAGE AND DOWNLOAD
# ==============================================================================

print("="*80)
print(" "*25 + "SECTION 3: PACKAGE AND DOWNLOAD")
print("="*80 + "\n")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"EPLB_Paper_Results_{timestamp}.zip"

print(f"Creating ZIP archive: {zip_filename}")
print("-"*80 + "\n")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:

    # Add Tables
    print("📊 Adding Tables:")
    zipf.write('table_i_instance_specs.csv', 'Table_I_Instance_Specs.csv')
    print("  ✓ Table I: Instance Specifications")

    zipf.write('table_ii_main_results.csv', 'Table_II_Main_Results.csv')
    print("  ✓ Table II: Main Experimental Results")

    # Add Diagrams
    print("\n📈 Adding Diagrams:")
    zipf.write('diagram_1_energy_comparison.png', 'Diagram_1_Energy_Comparison.png')
    print("  ✓ Diagram 1: Energy Consumption Comparison")

    zipf.write('diagram_2_latency_boxplot.png', 'Diagram_2_Latency_Distribution.png')
    print("  ✓ Diagram 2: Latency Distribution")

    zipf.write('diagram_3_instance_utilization.png', 'Diagram_3_Instance_Utilization.png')
    print("  ✓ Diagram 3: Instance Type Utilization")

    zipf.write('diagram_4_efficiency_score.png', 'Diagram_4_Efficiency_Score.png')
    print("  ✓ Diagram 4: Efficiency Score Comparison")

    # Add raw data
    print("\n📁 Adding Raw Data:")
    zipf.write('simulation_results.csv', 'Raw_Simulation_Results.csv')
    print("  ✓ Complete simulation results")

print("\n" + "="*80)
print(f"✅ SUCCESS! ZIP ARCHIVE CREATED: {zip_filename}")
print("="*80)

print("\n📦 Archive Contents:")
print("  • 2 Tables (CSV format)")
print("    - Table I: Instance pool configuration and specifications")
print("    - Table II: Main results (3 algorithms × 2 workloads)")
print("\n  • 4 Diagrams (PNG format, 300 DPI publication quality)")
print("    - Diagram 1: Energy comparison bar chart")
print("    - Diagram 2: Latency distribution box plots")
print("    - Diagram 3: Instance utilization stacked bars")
print("    - Diagram 4: Efficiency score comparison (NEW)")
print("\n  • 1 Raw Data File (Complete simulation results)")

file_size_kb = os.path.getsize(zip_filename) / 1024
print(f"\n📊 File size: {file_size_kb:.1f} KB")
print(f"📂 Total files packaged: 7")
print("="*80 + "\n")

# Trigger download in Google Colab
try:
    from google.colab import files
    files.download(zip_filename)
    print("🔽 Download started automatically (Google Colab)")
    print("    Check your browser's download folder")
except ImportError:
    print(f"💡 Download '{zip_filename}' manually from the file browser")
    print("    (Left sidebar → Files → Right-click → Download)")

print("\n" + "="*80)
print("🎉 SIMULATION COMPLETE!")
print("="*80)

print("\n📋 Key Results Summary:")
print("-"*80)

# Calculate energy savings
eplb_energy = results_df[results_df['algorithm'] == 'EPLB']['total_energy_kwh'].mean()
rr_energy = results_df[results_df['algorithm'] == 'RoundRobin']['total_energy_kwh'].mean()
lc_energy = results_df[results_df['algorithm'] == 'LeastConnections']['total_energy_kwh'].mean()
savings_vs_rr = ((rr_energy - eplb_energy) / rr_energy) * 100
savings_vs_lc = ((lc_energy - eplb_energy) / lc_energy) * 100

print(f"💡 ENERGY PERFORMANCE:")
print(f"  EPLB Average Energy:           {eplb_energy:.2f} kWh")
print(f"  Round Robin Average Energy:    {rr_energy:.2f} kWh")
print(f"  LeastConnections Average:      {lc_energy:.2f} kWh")
print(f"  \n  Energy Savings vs Round Robin: {savings_vs_rr:.1f}%")
print(f"  Energy Savings vs LeastConn:   {savings_vs_lc:.1f}%")

# Latency comparison
eplb_latency = results_df[results_df['algorithm'] == 'EPLB']['p95_latency_ms'].mean()
rr_latency = results_df[results_df['algorithm'] == 'RoundRobin']['p95_latency_ms'].mean()
lc_latency = results_df[results_df['algorithm'] == 'LeastConnections']['p95_latency_ms'].mean()

print(f"\n⚡ LATENCY PERFORMANCE:")
print(f"  EPLB Average p95 Latency:      {eplb_latency:.2f} ms")
print(f"  Round Robin Average p95:       {rr_latency:.2f} ms")
print(f"  LeastConnections Average p95:  {lc_latency:.2f} ms")

latency_improvement_rr = ((rr_latency - eplb_latency) / rr_latency) * 100
print(f"  \n  Latency Improvement vs RR:     {latency_improvement_rr:.1f}%")

# Instance distribution
eplb_avg = results_df[results_df['algorithm'] == 'EPLB'][['cpu_usage_pct', 'l4_usage_pct', 'h100_usage_pct']].mean()
print(f"\n🖥️  EPLB INSTANCE DISTRIBUTION:")
print(f"  CPU:  {eplb_avg['cpu_usage_pct']:5.1f}%")
print(f"  L4:   {eplb_avg['l4_usage_pct']:5.1f}%")
print(f"  H100: {eplb_avg['h100_usage_pct']:5.1f}%")

# Efficiency scores
eplb_eff = eff_df[eff_df['Algorithm'] == 'EPLB']['Efficiency Score'].mean()
rr_eff = eff_df[eff_df['Algorithm'] == 'RoundRobin']['Efficiency Score'].mean()
lc_eff = eff_df[eff_df['Algorithm'] == 'LeastConnections']['Efficiency Score'].mean()

print(f"\n📊 OVERALL EFFICIENCY SCORES:")
print(f"  EPLB:             {eplb_eff:6.2f} (best)")
print(f"  Round Robin:      {rr_eff:6.2f}")
print(f"  LeastConnections: {lc_eff:6.2f}")

efficiency_improvement = ((eplb_eff - rr_eff) / rr_eff) * 100
print(f"  \n  Efficiency Improvement:        {efficiency_improvement:.1f}%")

print("\n" + "="*80)
print("✨ All outputs ready for paper submission!")
print("="*80)

print("\n📝 WHAT TO DO NEXT:")
print("-"*80)
print("1. ✓ Download the ZIP file (should start automatically)")
print("2. ✓ Extract: Table_I_Instance_Specs.csv")
print("3. ✓ Extract: Table_II_Main_Results.csv")
print("4. ✓ Extract: All 4 diagram PNG files")
print("5. ✓ Insert tables and diagrams into your paper")
print("\n6. 📄 UPDATE YOUR PAPER:")
print("   - Remove all references to 'Table III'")
print("   - Remove all references to 'Diagram 5' or 'Figure 5'")
print("   - Remove Section V.C (Sensitivity Analysis)")
print("   - Update any mentions of 'L_max = 50ms' or 'L_max = 200ms'")
print("   - Add description of new Diagram 4 (Efficiency Score)")
print("\n7. 🔍 SEARCH & REPLACE IN PAPER:")
print("   - Search: 'sensitivity' → Delete all mentions")
print("   - Search: 'Table III' → Delete all references")
print("   - Search: 'Figure 5' OR 'Diagram 5' → Delete all references")
print("   - Search: '50ms' → Remove (unless used elsewhere)")
print("   - Search: '200ms' → Remove (unless used elsewhere)")
print("="*80 + "\n")

print("🎯 PAPER UPDATES FOR DIAGRAM 4:")
print("-"*80)
print("Add this to your paper when describing Diagram 4:")
print("\nDiagram 4 presents an overall efficiency score that combines throughput,")
print("energy consumption, and latency into a single metric. The efficiency score")
print("is calculated as: Score = Throughput / (Energy × Latency), where higher")
print("scores indicate better overall system performance. EPLB achieves efficiency")
print(f"scores of {eplb_eff:.1f} on average, representing a {efficiency_improvement:.1f}% improvement")
print("over Round Robin. This demonstrates that EPLB simultaneously optimizes")
print("multiple performance dimensions rather than trading off one for another.")
print("="*80 + "\n")

                         SECTION 1: RUN EXPERIMENTS

Instance Pool Configuration (Moderate Capacity Ratios 1:3:6):
  CPU  : 20 instances ×  50 req/sec = 1000 req/sec total capacity
  L4   : 10 instances × 150 req/sec = 1500 req/sec total capacity
  H100 :  5 instances × 300 req/sec = 1500 req/sec total capacity

  TOTAL SYSTEM CAPACITY: 4000 req/sec

Running experiments...
--------------------------------------------------------------------------------
EPLB               + Sinusoidal   ... ✓
EPLB               + Spiky        ... ✓
RoundRobin         + Sinusoidal   ... ✓
RoundRobin         + Spiky        ... ✓
LeastConnections   + Sinusoidal   ... ✓
LeastConnections   + Spiky        ... ✓

✓ Completed in 49.7 seconds
✓ 6 experiments executed successfully

Sample Results:
       algorithm   workload  total_energy_kwh  p95_latency_ms  cpu_usage_pct  l4_usage_pct  h100_usage_pct
            EPLB Sinusoidal         38.929048       18.800000       0.000000     94.266416        5.733584
     

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🔽 Download started automatically (Google Colab)
    Check your browser's download folder

🎉 SIMULATION COMPLETE!

📋 Key Results Summary:
--------------------------------------------------------------------------------
💡 ENERGY PERFORMANCE:
  EPLB Average Energy:           41.48 kWh
  Round Robin Average Energy:    82.87 kWh
  LeastConnections Average:      83.39 kWh
  
  Energy Savings vs Round Robin: 49.9%
  Energy Savings vs LeastConn:   50.3%

⚡ LATENCY PERFORMANCE:
  EPLB Average p95 Latency:      18.80 ms
  Round Robin Average p95:       118.87 ms
  LeastConnections Average p95:  118.80 ms
  
  Latency Improvement vs RR:     84.2%

🖥️  EPLB INSTANCE DISTRIBUTION:
  CPU:    0.3%
  L4:    88.2%
  H100:  11.5%

📊 OVERALL EFFICIENCY SCORES:
  EPLB:             10548.71 (best)
  Round Robin:      938.41
  LeastConnections: 936.06
  
  Efficiency Improvement:        1024.1%

✨ All outputs ready for paper submission!

📝 WHAT TO DO NEXT:
---------------------------------------------------